# Dairy milk-yield prediction: interpretable vs heavy models

A narrated walkthrough of the analysis in this repository. It mirrors what
`python -m src.run_analysis` does, but step by step with explanation.

**Question:** on a realistic dairy dataset of a few hundred records, do simple
interpretable models lose much accuracy to heavier tree-ensemble models?

> The data here is a clearly-labelled synthetic placeholder unless the real
> CSV is present (see the README). Swap in the real data and re-run to get
> real numbers.

In [1]:
import sys
from pathlib import Path

# make the src package importable when running inside notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd

from src.data import FEATURES, TARGET, get_xy, load_data
from src.models import INTERPRETABLE, build_models, param_grids
from src.evaluate import evaluate_all

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 1. Load the data

`load_data()` returns the real CSV if it exists, otherwise a faithful
placeholder whose per-lactation statistics match the published dataset.

In [2]:
df, is_placeholder = load_data()
print("PLACEHOLDER (synthetic)" if is_placeholder else "REAL data")
print(f"{len(df)} rows, {df.shape[1]} columns")
df.head()

PLACEHOLDER (synthetic)
240 rows, 7 columns


,cow_id,parity,lactation_length_d,peak_day,peak_milk_kg,dry_period_d,milk_yield_kg
0,FH-1-000,1,256.500,55.400,33.820,63.600,"6,714.800"
1,FH-1-001,1,308.400,90.600,28.220,53.400,"6,660.200"
2,FH-1-002,1,327.500,91.800,35.670,47.500,"8,012.500"
3,FH-1-003,1,343.800,125.100,36.460,61.700,"8,071.300"
4,FH-1-004,1,348.600,109.800,24.430,55.400,"7,506.400"


A quick look at the target and predictors.

In [3]:
df[[TARGET] + FEATURES].describe()

,milk_yield_kg,parity,lactation_length_d,peak_day,peak_milk_kg,dry_period_d
count,240.000,240.000,240.000,240.000,240.000,240.000
mean,"7,766.008",2.500,319.123,68.716,40.601,61.718
std,"1,153.321",1.120,39.323,26.675,6.882,19.132
min,"4,598.900",1.000,206.800,10.100,23.100,20.000
25%,"6,953.300",1.750,293.025,51.125,35.163,48.525
50%,"7,808.600",2.500,319.650,66.650,41.270,60.700
75%,"8,634.875",3.250,346.600,83.900,45.930,73.175
max,"10,728.100",4.000,438.000,170.600,56.650,120.000


## 2. Features and target

We predict total lactation milk yield (kg) from parity, lactation length, peak
day, peak milk, and dry-period length.

In [4]:
X, y = get_xy(df)
print("features:", list(X.columns))
print("target  :", TARGET)

features: ['parity', 'lactation_length_d', 'peak_day', 'peak_milk_kg', 'dry_period_d']
target  : milk_yield_kg


## 3. The models

Three interpretable (Linear Regression, Elastic Net, SVR) and two heavy
(Random Forest, Gradient Boosting). Scale-sensitive models are scaled inside a
pipeline so the comparison is fair.

In [5]:
models = build_models()
grids = param_grids()
for name in models:
    kind = "interpretable" if name in INTERPRETABLE else "heavy"
    print(f"  {name:18s} [{kind}]")

  Linear Regression  [interpretable]
  Elastic Net        [interpretable]
  SVR                [interpretable]
  Random Forest      [heavy]
  Gradient Boosting  [heavy]


## 4. Evaluate with nested cross-validation

Why nested CV: tuning and scoring on the same folds leaks information and
flatters the heavy models. The inner loop tunes; the outer loop scores. Every
number below is an out-of-fold estimate.

In [6]:
results, oof_preds = evaluate_all(models, grids, X, y)
results

,RMSE,MAE,R2
model,,,
Gradient Boosting,667.774,529.547,0.663
Elastic Net,684.363,552.711,0.646
Linear Regression,684.570,552.359,0.646
Random Forest,689.565,540.495,0.641
SVR,902.260,728.829,0.385


## 5. Read the result

Compare the best interpretable model against the best model overall.

In [7]:
best = results.index[0]
best_interp = next(n for n in results.index if n in INTERPRETABLE)
gap = results.loc[best_interp, "RMSE"] - results.loc[best, "RMSE"]

print(f"best overall      : {best} (RMSE {results.loc[best, 'RMSE']:,.0f} kg)")
print(f"best interpretable: {best_interp} (RMSE {results.loc[best_interp, 'RMSE']:,.0f} kg)")
print(f"gap               : {gap:,.0f} kg "
      f"({100 * gap / results.loc[best, 'RMSE']:.1f}% of best RMSE)")

best overall      : Gradient Boosting (RMSE 668 kg)
best interpretable: Elastic Net (RMSE 684 kg)
gap               : 17 kg (2.5% of best RMSE)


**Takeaway.** When the gap between the best heavy model and the best
interpretable model is only a few percent, the interpretable model is usually
the better scientific choice: you keep almost all the accuracy and gain a model
you can actually explain and defend. On small animal-science datasets this is
the common situation, which is the argument this project is built to make.

Re-run with the real dataset (see the README) to replace these placeholder
numbers with real ones.